# Google Places Debug Notebook

Step-by-step checks for `src/sites/google.py`.

Covers:
1. **Playwright scraping** (Plan A) – Google Maps rating via Playwright
2. **API call** (Plan B) – Google Places API (requires `GOOGLE_MAPS_API_KEY`)

**Requirements:** Playwright installed. `GOOGLE_MAPS_API_KEY` for API fallback.

In [ ]:
from pathlib import Path
import sys
import os

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)


In [ ]:
from src.sites import google as site

In [ ]:
api_key = os.getenv('GOOGLE_MAPS_API_KEY')
print(f'API key: {"set" if api_key else "NOT SET (Playwright-only mode)"}')

## 1. Playwright – Google Maps Rating (Plan A)

Scrapes the overall Google rating directly from Google Maps. No API key needed.

In [ ]:
from src.sites.google_scraper import fetch_google_rating_playwright, GOOGLE_MAPS_URLS

# Test Playwright rating for a single hotel
hotel, maps_url = next(iter(GOOGLE_MAPS_URLS.items()))
print(f'Testing: {hotel}')
print(f'URL: {maps_url}')
print()

rating, num_reviews = fetch_google_rating_playwright(maps_url)
print(f'Rating: {rating}/5 ({num_reviews} reviews)')

In [ ]:
# Fetch Playwright rating for ALL hotels
print('Google Maps Playwright ratings for all hotels:')
print()
for hotel, maps_url in GOOGLE_MAPS_URLS.items():
    try:
        rating, num_reviews = fetch_google_rating_playwright(maps_url)
        print(f'  {hotel:45s}  {rating}/5  ({num_reviews} reviews)')
    except Exception as exc:
        print(f'  {hotel:45s}  ERROR: {exc}')

## 2. API – Google Places Rating (Plan B / Fallback)

Requires `GOOGLE_MAPS_API_KEY`.

In [ ]:
site.HOTEL_QUERIES

In [ ]:
hotel, query = next(iter(site.HOTEL_QUERIES.items()))
print('Testing:', hotel)
score = site.get_google_rating(query, api_key=api_key, timeout=20)
print('Score:', score)


In [ ]:
results = {}
for hotel, query in site.HOTEL_QUERIES.items():
    try:
        results[hotel] = site.get_google_rating(query, api_key=api_key, timeout=20)
    except Exception as exc:
        results[hotel] = f'ERROR: {exc}'
results


In [ ]:
from datetime import datetime
import subprocess

date_col = datetime.now().strftime('%Y-%m-%d')
cmd = [sys.executable, str(ROOT / 'src' / 'sites' / 'google.py'), '--date', date_col]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=False)
